In [ ]:
# from yandex_cloud_ml_sdk import YCloudML

# sdk = YCloudML(folder_id="folder_id", auth="auth")

# model = sdk.models.completions("yandexgpt-4-lite")
# model = model.configure(temperature=0.5)
# result = model.run("Что такое небо?")

# for alternative in result:
#     print(alternative)

Alternative(role='assistant', text='Небо — это пространство над поверхностью Земли или другого астрономического объекта. С поверхности Земли небо выглядит как свод, по которому движутся Солнце, Луна, звёзды и различные небесные тела.\n\nНебо может иметь разный цвет в зависимости от времени суток, погодных условий и наличия в атмосфере частиц пыли или влаги. Например, днём небо обычно имеет голубой цвет из-за рассеяния солнечного света в атмосфере, ночью же оно становится тёмным, поскольку исчезает источник света — Солнце.', status=<AlternativeStatus.FINAL: 3>, tool_calls=None)


In [86]:
SYSTEM_PROMPT = """\
Ты специалист по разметке данных. Твоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:
1. Составить краткую текстовую сводку (summarized_review).
2. Определить все темы и подтемы ("topic"), о которых упоминает клиент. Это в основном продукты/услуги банка.
3. Для каждой упомянутой темы определить тональность высказывания ("sentiment"): positive, negative или neutral.

**ВАЖНЫЕ ИНСТРУКЦИИ:**
- **Темы:** Используй ТОЛЬКО следующий список тем и подтем. Не добавляй новые темы. Если в тексте присутствует подтема, то добавь и ещё и основную тему.
- **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без эмоциональной окраски.
- **Строгость:** Не выдумывай темы. Если в отзыве нет явного упоминания продукта или услуги, не включай его.
- **Символы `****`:** Это конфиденциальные данные или ненормативная лексика. Учитывай общий контекст.
- **Если темы нет:** Если невозможно определить ни одну тему, верни пустой массив `topic_sentiment_pairs`.

**ДОПУСТИМЫЕ ТЕМЫ:**
- Офисное обслуживание (это обслуживание в отделениях банка)
- Дистанционное обслуживание (это звонки, чаты, онлайн-консультации и подобное)
- Банкоматы
- Курьерская доставка карт
- Обмен валют
- Дебетовые карты (включая подтемы: Денежные переводы, Карта UnionPay, Умная дебетовая карта «Мир», Премиальная карта Mir Supreme)
- Кредитные карты (включая подтемы: Кредитная карта 180 дней Премиум)
- Кредиты (включая подтемы: Кредит наличными, Кредит наличными под залог недвижимости, Кредит под залог автомобиля)
- Рефинансирование/Реструктуризация (включая подтемы: Рефинансирование кредитов, Реструктуризация кредитов, Рефинансирование ипотеки, Реструктуризация ипотеки)
- Автокредиты
- Ипотека
- Страховые и сервисные продукты
- Вклады (включая подтемы: Вклад «Копить», Вклад «В Плюсе», Вклад «Новые деньги»)
- Накопительные счета (включая подтемы: Накопительный счёт «Ежедневная выгода», Накопительный счёт «Ежедневный процент», Накопительный счёт «Премиум»)
- Акции и бонусы (включая подтемы: Газпром Бонус, Газпромбанк Привилегии, Кэшбэк, Акции, Программы лояльности)
- Газпромбанк Премиум (включая подтемы: Персональный менеджер, Консьерж-сервис, Премиальное обслуживание)
- Мобильное приложение
- Другие услуги банка (включая подтемы: Газпромбанк Travel (покупка авиабилетов/отелей), Gazprom Pay (оплата телефоном), GorodPay (оплата общественного транспорта), Инвестиционные продукты, Брокерские услуги, Депозитарные услуги, Аренда сейфовых ячеек)

**Формат ответа — строго JSON:**
[
    {
        "id": "исходный_id_отзыва",
        "summarized_review": "Краткая сводка отзыва на русском языке.",
        "topic_sentiment_pairs": [
            {
                "topic": "Название темы 1",
                "sentiment": "positive/negative/neutral"
            },
            {
                "topic": "Название темы 2", 
                "sentiment": "positive/negative/neutral"
            }
        ]
    }
]

**ПРИМЕРЫ:**

Отзыв 1: "Мобильное приложение постоянно зависает при переводе денег. Зато вклад 'Копить' открыла без проблем, процентная ставка радует."

Отзыв 2: "Курьер опоздал на 2 часа с доставкой карты, даже не позвонил. Пытался дозвониться в поддержку - 40 минут на линии, так и не дождался ответа."

Отзыв 3: "Оформил Премиальную карту Mir Supreme через мобильное приложение - очень удобно. Также пользуюсь Газпромбанк Travel для бронирования отелей."

Ответ:
[
    {
        "id": "1",
        "summarized_review": "Клиент жалуется на нестабильную работу мобильного приложения при переводах, но доволен процессом открытия и условиями вклада 'Копить'.",
        "topic_sentiment_pairs": [
            {
                "topic": "Мобильное приложение",
                "sentiment": "negative"
            },
            {
                "topic": "Денежные переводы", 
                "sentiment": "neutral"
            },
            {
                "topic": "Вклад «Копить»",
                "sentiment": "positive"
            },
            {
                "topic": "Вклады",
                "sentiment": "positive"
            }
        ]
    },
    {
        "id": "2", 
        "summarized_review": "Клиент сообщает о проблемах с доставкой карты (курьер опоздал) и невозможности дозвониться в службу поддержки.",
        "topic_sentiment_pairs": [
            {
                "topic": "Доставка карт",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": "3",
        "summarized_review": "Клиент положительно оценивает оформление премиальной карты через приложение и говорит об использовании сервиса Газпромбанк Travel.",
        "topic_sentiment_pairs": [
            {
                "topic": "Премиальная карта Mir Supreme",
                "sentiment": "neutral"
            },
            {
                "topic": "Дебетовые карты",
                "sentiment": "neutral"
            },
            {
                "topic": "Газпромбанк Премиум", 
                "sentiment": "neutral"
            },
            {
                "topic": "Мобильное приложение",
                "sentiment": "positive"
            },
            {
                "topic": "Газпромбанк Travel",
                "sentiment": "neutral"
            },
            {
                "topic": "Другие услуги банка",
                "sentiment": "neutral"
            }
        ]
    }
]

**Вот отзывы клиентов, которые необходимо обработать:**
"""

In [87]:
import json
import pandas as pd
import numpy as np
# from collections import Counter
# import matplotlib.pyplot as plt
# import seaborn as sns

In [88]:
df = pd.read_csv("data/data_latest.csv")

df

,review_id,date,review_text,topic,subtopic,sentiment
0,1000087,2025-09-19,Вклад «Новые деньги» невозможно оформить без п...,Вклады,NaN,Negative
1,999494,2025-09-18,В июне 2025 года я порекомендовал премиальную ...,Дебетовые карты,NaN,Negative
2,999142,2025-09-17,Мошенниччиские аперации в интересах Ренессанс ...,Обслуживание,NaN,Negative
3,998360,2025-09-15,Купил услугу Газпром Бонус «Премиум» за 2 990 ...,Дебетовые карты,NaN,Negative
4,998516,2025-09-15,Производил оформление открытия срочного банков...,Вклады,«Накопительный»,Negative
...,...,...,...,...,...,...
4773,7470,2011-04-07,Ужастное обслуживание! Мало того потеряли доку...,Обслуживание,NaN,Negative
4774,7049,2011-03-28,Могут заблокировать рассчетную или кредитную к...,Кредитные карты,NaN,Negative
4775,5221,2011-01-25,"Мало того уже прошла неделя, а ПТС так и не ве...",Автокредиты,NaN,Negative
4776,5053,2011-01-16,Газпромбанк– отличный банк с отличными сотрудн...,Ипотека,NaN,Positive


In [34]:
df_randomised = df.sample(df.shape[0], random_state=42)

In [100]:
df_start = 20
df_end = df_start + 20
df_start, df_end

(20, 40)

In [101]:
df_part_randomised = df_randomised.iloc[df_start:df_end].copy()

df_part_randomised["review_text"] = df_part_randomised["review_text"].str.replace("\n", " ")

texts = df_part_randomised.apply(lambda x: f"Отзыв {x['review_id']}: {x['review_text']}", axis=1)

combined_text = "\n\n".join(texts.values)

In [102]:
print(combined_text)

Отзыв 409556: Как и все. Кредит от 5,5%, но Вам спец предложение 13,5%. Без лоха и жизнь плоха! Так продолжать!

Отзыв 405166: Включили платные СМС уведомления, снять в приложении техническая возможность заблокирована, месяц разбирательств, никакого результата, теперь и приложение заблокировано и нет возможности входа в него, обслуживание и тех поддержка на первобытном уровне, последний мой опыт работы с этим банком!

Отзыв 416697: Здравствуйте. Мне нужно было поменять карту, истек срок действия. Я обратилась в ближайший филиал, но оказалось, что карта находится в другом филиале, хотя когда я меняла карту в прошлый раз, получала в этом. Уже дважды я ездила в тот второй филиал, но так и не получила карту, поскольку там была очередь 15-20 человек, а у меня двое маленьких детей, и я не могла ждать столько времени. По телефону мне не удалось добиться, чтобы новую карту перевезли в удобный для меня филиал.

Отзыв 367491: С 16.05 Жду решения банка по рефинансирование кредита. Кредит в Газпро

In [ ]:
import requests

# modelUri = "gpt://b1g92tujam7if119qep9/yandexgpt-4-lite/latest"
# modelUri = "gpt://b1g92tujam7if119qep9/yandexgpt-5.1/latest"
modelUri = "gpt://b1g92tujam7if119qep9/qwen3-235b-a22b/latest"
# modelUri = "gpt://b1g92tujam7if119qep9/qwen3-14b/latest"
# modelUri = "gpt://b1g92tujam7if119qep9/gemma-3-27b-it/latest"
# modelUri = "gpt://b1g92tujam7if119qep9/gpt-oss-120b/latest"
# modelUri = "gpt://b1g92tujam7if119qep9/llama3.3-70b-instruct/latest"


prompt = {
    "modelUri": modelUri,
    "completionOptions": {
        "stream": False,
        "temperature": 0.1,
        "maxTokens": "2000"
    },
    "messages": [
        {
            "role": "system",
            "text": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "text": combined_text
        }
    ]
}


url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
headers = {
    "Content-Type": "application/json",
    "Authorization": "Api-Key Api-Key"
}

response = requests.post(url, headers=headers, json=prompt)
result = response.text
print(result)

{"error":{"grpcCode":13,"httpCode":500,"message":"Fatal internal error in TextGenerationService.Completion","httpStatus":"Internal Server Error","details":[]}}


In [113]:
df["review_text"][df["review_text"].apply(len).argsort()].iloc[-250:]

777     Я участник СВО, получаю зарплату на зарплатную...
172     02.05.2025 Мной был сделан перевод моему родст...
371     Пытался перевести деньги из вашего банка себе ...
4352    Добрый день, я очень недовольна службой безопа...
4760    Мне отправили значительную сумму переводом из–...
                              ...                        
1173    -Алло! Здравствуйте, вы должны нам 280 тысяч р...
769     25.09.2024 Года мной была отправлена заявка на...
580     Добрый вечер! 03.11.24 С моей кредитной карты ...
973     В начале июля 2024 года по реферальной ссылке ...
662     Решила окунуться в премиальное обслуживание, к...
Name: review_text, Length: 250, dtype: object

In [104]:
result = json.loads(result)

In [105]:
print(result["result"]["alternatives"][0]["message"]["text"])

[
    {
        "id": "409556",
        "summarized_review": "Клиент недоволен тем, что банк предлагает ему кредит под 13,5% годовых, в то время как обычная ставка составляет 5,5%.",
        "topic_sentiment_pairs": [
            {
                "topic": "Кредиты",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": "405166",
        "summarized_review": "Клиент недоволен тем, что банк включил платные СМС-уведомления без его согласия, и теперь он не может снять их в приложении. Также он недоволен работой поддержки и обслуживания.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Мобильное приложение",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": "416697",
        "summarized_review": "Клиентке необходимо было поменять карту, но она не смогла получить е